# Sistema de Monitoreo de Calidad del Agua

Modelo de Machine Learning con Random Forest para detectar contaminación en fuentes hídricas
usando datos de sensores de pH, turbidez y temperatura.

- pH
- turbidez
- temperatura

## Estructura del proyecto

```
water_quality_monitor/
├── data/
│   ├── raw/            # Dataset original de Kaggle (sin modificar)
│   └── processed/      # Datos limpios y listos para entrenar
├── models/             # Modelo entrenado guardado (.pkl)
├── notebooks/          # Exploración y análisis del dataset
├── src/
│   ├── preprocess.py   # Limpieza y preparación de datos
│   ├── train.py        # Entrenamiento del modelo
│   ├── evaluate.py     # Evaluación y métricas
│   └── predict.py      # Predicciones con nuevas muestras
├── reports/            # Gráficos y resultados exportados
├── requirements.txt
└── README.md
```

## Instalación

```bash
# 1. Instalar los Requerimientos
pip install -r requirements.txt
```

## Configurar Jupyter y Entorno Virtual (venv)
1. Instalar Extension de Jupyter en VSCode
```bash
https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter
```

2. Instalar Jupyter en Python 
```bash
pip install jupyter notebook ipykernel
```

3. Seleccionar un Kernel (Entorno de Pyton)
    - Abre o crea un archivo .ipynb
    - En la esquina superior derecha verás "Select Kernel"
    - Elige tu intérprete de Python (o un entorno virtual si usas uno)

## Uso rápido (RAW)

```bash
# 1. Preprocesar los datos
python src/preprocess.py

# 2. Entrenar el modelo
python src/train.py

# 3. Evaluar resultados
python src/evaluate.py

# 4. Predecir una nueva muestra
python src/predict.py
```


# Arquitectura del Modelo

![mi imagen](./resources/random_forest_flow.svg)



#### Preprocess.py
Limpieza y preparación del dataset de calidad del agua.
Ajusta las columnas según el dataset que descargues de Kaggle.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Rutas
RAW_PATH       = Path("data/raw/water_quality.csv")   # <-- cambia por tu archivo
PROCESSED_PATH = Path("data/processed/water_clean.csv")


def load_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
    print(df.head())
    return df


def inspect_data(df: pd.DataFrame):
    """Muestra información básica: tipos, nulos, estadísticas."""
    print("\n--- Tipos de datos ---")
    print(df.dtypes)
    print("\n--- Valores nulos ---")
    print(df.isnull().sum())
    print("\n--- Estadísticas descriptivas ---")
    print(df.describe())


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Limpieza básica:
    - Elimina duplicados
    - Rellena valores nulos con la mediana de cada columna
    """
    df = df.drop_duplicates()

    # Rellenar nulos con la mediana (más robusto que la media para datos de sensores)
    for col in df.select_dtypes(include=[np.number]).columns:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)

    print(f"\nDataset limpio: {df.shape[0]} filas")
    return df


def select_features(df: pd.DataFrame, feature_cols: list, target_col: str) -> tuple:
    """
    Selecciona las columnas de sensores (features) y la columna objetivo (target).

    Parámetros:
        feature_cols : lista con los nombres de las columnas de sensores
                       Ejemplo: ['ph', 'Turbidity', 'Temperature']
        target_col   : nombre de la columna con la etiqueta
                       Ejemplo: 'Potability' o 'is_safe'
    """
    X = df[feature_cols]
    y = df[target_col]
    print(f"\nFeatures: {feature_cols}")
    print(f"Target: {target_col} | Distribución:\n{y.value_counts()}")
    return X, y


if __name__ == "__main__":
    # ---------------------------------------------------------------
    # AJUSTA ESTAS VARIABLES según las columnas de tu dataset Kaggle
    # ---------------------------------------------------------------
    FEATURE_COLUMNS = ["ph", "Turbidity", "Temperature"]  # <-- nombres reales del CSV
    TARGET_COLUMN   = "Potability"                         # <-- columna de etiqueta

    df = load_data(RAW_PATH)
    inspect_data(df)
    df = clean_data(df)

    X, y = select_features(df, FEATURE_COLUMNS, TARGET_COLUMN)

    # Guardar datos procesados
    PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
    clean_df = X.copy()
    clean_df[TARGET_COLUMN] = y.values
    clean_df.to_csv(PROCESSED_PATH, index=False)
    print(f"\nDatos guardados en: {PROCESSED_PATH}")


ModuleNotFoundError: No module named 'pandas'

## Modelos de la red

```
src/
├── models/
│   ├── train.py        # Entrenamiento del modelo
│   ├── evaluate.py     # Evaluación y métricas
│   └── predict.py      # Predicciones con nuevas muestras
└── main.py
```

#### Evaluate.py
Evaluación del modelo entrenado: métricas, matriz de confusión e importancia de variables.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, roc_auc_score, roc_curve
)

MODEL_PATH    = Path("models/random_forest.pkl")
REPORTS_DIR   = Path("reports")
TARGET_COLUMN = "Potability"
CLASS_NAMES   = ["Potable", "No potable"]   # ajusta si tu dataset tiene otros nombres


def load_test_data():
    X_test = pd.read_csv("data/processed/X_test.csv")
    y_test = pd.read_csv("data/processed/y_test.csv").squeeze()
    return X_test, y_test


def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print("=" * 50)
    print("RESULTADOS DEL MODELO")
    print("=" * 50)
    print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
    print(f"F1-score : {f1_score(y_test, y_pred):.4f}")
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}")
    print("\nReporte completo:")
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

    return y_pred, y_prob


def plot_confusion_matrix(y_test, y_pred):
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Real")
    ax.set_title("Matriz de Confusión — Calidad del Agua")
    plt.tight_layout()
    REPORTS_DIR.mkdir(exist_ok=True)
    path = REPORTS_DIR / "confusion_matrix.png"
    plt.savefig(path, dpi=150)
    print(f"Gráfico guardado: {path}")
    plt.show()


def plot_feature_importance(model, feature_names):
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]

    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.barh(
        [feature_names[i] for i in indices],
        importances[indices],
        color="#3B8BD4"
    )
    ax.set_xlabel("Importancia")
    ax.set_title("Importancia de Variables — Random Forest")
    ax.invert_yaxis()

    for bar, val in zip(bars, importances[indices]):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
                f"{val:.3f}", va="center", fontsize=10)

    plt.tight_layout()
    path = REPORTS_DIR / "feature_importance.png"
    plt.savefig(path, dpi=150)
    print(f"Gráfico guardado: {path}")
    plt.show()


def plot_roc_curve(y_test, y_prob):
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, color="#1D9E75", lw=2, label=f"ROC (AUC = {auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("Tasa de Falsos Positivos")
    ax.set_ylabel("Tasa de Verdaderos Positivos")
    ax.set_title("Curva ROC — Detección de Contaminación")
    ax.legend(loc="lower right")
    plt.tight_layout()
    path = REPORTS_DIR / "roc_curve.png"
    plt.savefig(path, dpi=150)
    print(f"Gráfico guardado: {path}")
    plt.show()


if __name__ == "__main__":
    model  = joblib.load(MODEL_PATH)
    X_test, y_test = load_test_data()
    y_pred, y_prob = evaluate(model, X_test, y_test)
    plot_confusion_matrix(y_test, y_pred)
    plot_feature_importance(model, list(X_test.columns))
    plot_roc_curve(y_test, y_prob)


#### Predict.py
Realiza predicciones con el modelo entrenado sobre nuevas muestras de sensores.
Simula los valores que llegarían de sensores físicos en tiempo real.


In [ ]:
import numpy as np
import joblib
from pathlib import Path

MODEL_PATH  = Path("models/random_forest.pkl")
SCALER_PATH = Path("models/scaler.pkl")

# Nombres de las features (deben coincidir con el orden del entrenamiento)
FEATURE_NAMES = ["ph", "Turbidity", "Temperature"]


def predict_sample(ph: float, turbidity: float, temperature: float) -> dict:
    """
    Recibe los valores de los tres sensores y devuelve la predicción.

    Retorna un dict con:
        - prediction : 0 = potable, 1 = no potable
        - label      : texto legible
        - probability: probabilidad de contaminación (0.0 a 1.0)
    """
    model  = joblib.load(MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)

    sample = np.array([[ph, turbidity, temperature]])
    sample_scaled = scaler.transform(sample)

    prediction  = model.predict(sample_scaled)[0]
    probability = model.predict_proba(sample_scaled)[0][1]

    label = "NO POTABLE ⚠️" if prediction == 1 else "POTABLE ✓"

    return {
        "prediction":  int(prediction),
        "label":       label,
        "probability": round(float(probability), 4)
    }


if __name__ == "__main__":
    # Ejemplos de prueba — ajusta los valores según los rangos de tu dataset
    samples = [
        {"ph": 7.0,  "turbidity": 2.1,  "temperature": 22.0},  # agua normal
        {"ph": 4.2,  "turbidity": 15.8, "temperature": 31.0},  # posible contaminación
        {"ph": 8.5,  "turbidity": 0.8,  "temperature": 18.0},  # agua limpia
    ]

    print("=" * 55)
    print("PREDICCIONES — SISTEMA DE CALIDAD DEL AGUA")
    print("=" * 55)

    for i, s in enumerate(samples, 1):
        result = predict_sample(s["ph"], s["turbidity"], s["temperature"])
        print(f"\nMuestra {i}:")
        print(f"  Entrada  → pH: {s['ph']} | Turbidez: {s['turbidity']} | Temp: {s['temperature']}°C")
        print(f"  Resultado → {result['label']}")
        print(f"  Probabilidad de contaminación: {result['probability']*100:.1f}%")


#### Train.py
Entrenamiento del modelo Random Forest para detección de contaminación.


In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler

# Rutas
PROCESSED_PATH = Path("data/processed/water_clean.csv")
MODEL_PATH     = Path("models/random_forest.pkl")
SCALER_PATH    = Path("models/scaler.pkl")

# Columna objetivo (debe coincidir con preprocess.py)
TARGET_COLUMN  = "Potability"


def load_processed_data():
    df = pd.read_csv(PROCESSED_PATH)
    X = df.drop(columns=[TARGET_COLUMN])
    y = df[TARGET_COLUMN]
    return X, y


def split_data(X, y, test_size=0.2, random_state=42):
    """
    Divide los datos: 80% entrenamiento, 20% prueba.
    random_state=42 asegura que siempre obtengas el mismo split (reproducibilidad).
    stratify=y mantiene la proporción de clases en ambos conjuntos.
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    print(f"Entrenamiento: {X_train.shape[0]} muestras")
    print(f"Prueba:        {X_test.shape[0]} muestras")
    return X_train, X_test, y_train, y_test


def scale_features(X_train, X_test):
    """
    Normaliza las features con StandardScaler.
    IMPORTANTE: fit solo en X_train, transform en ambos.
    Esto evita 'data leakage' (filtrar info del test al entrenamiento).
    """
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler


def train_model(X_train, y_train):
    """
    Crea y entrena el modelo Random Forest.

    Hiperparámetros:
      n_estimators : número de árboles (100 es un buen punto de partida)
      max_depth    : profundidad máxima de cada árbol (None = sin límite)
      random_state : semilla para reproducibilidad
      n_jobs       : núcleos de CPU a usar (-1 = todos)
    """
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    )

    print("\nEntrenando Random Forest...")
    model.fit(X_train, y_train)
    print("Entrenamiento completado.")

    # Validación cruzada (5 folds) — estimación más confiable del rendimiento
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="f1")
    print(f"\nValidación cruzada F1 (5 folds): {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

    return model


def save_artifacts(model, scaler):
    MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, MODEL_PATH)
    joblib.dump(scaler, SCALER_PATH)
    print(f"\nModelo guardado en:  {MODEL_PATH}")
    print(f"Scaler guardado en:  {SCALER_PATH}")


if __name__ == "__main__":
    X, y = load_processed_data()
    X_train, X_test, y_train, y_test = split_data(X, y)
    X_train_s, X_test_s, scaler = scale_features(X_train, X_test)
    model = train_model(X_train_s, y_train)
    save_artifacts(model, scaler)

    # Guardar X_test e y_test para usar en evaluate.py
    pd.DataFrame(X_test_s, columns=X.columns).to_csv("data/processed/X_test.csv", index=False)
    pd.Series(y_test.values, name=TARGET_COLUMN).to_csv("data/processed/y_test.csv", index=False)
    print("\nDatos de prueba guardados para evaluación.")
